In [11]:
import numpy as np
import pandas as pd
from pathlib import Path
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata
from sdv.utils import load_synthesizer


MODELS_DIR = Path("synthetic/models")

# ── validation ────────────────────────────────────────────────────────────────
def validate_dirs(real_data_dir, output_dir, ext='.txt'):
    real_data_dir = Path(real_data_dir)
    output_dir    = Path(output_dir)

    assert real_data_dir.exists(), \
        f"Real data directory not found: {real_data_dir}"

    files = list(real_data_dir.rglob(f"*{ext}"))
    assert len(files) > 0, \
        f"No {ext} files found in {real_data_dir}"

    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"✓ Found {len(files)} real files in {real_data_dir}")
    print(f"✓ Output directory ready: {output_dir}")
    return real_data_dir, output_dir

# ── model cache ───────────────────────────────────────────────────────────────
def fit_anemometer(real_data_dir):
    model_path    = MODELS_DIR / "anemometer_synthesizer.pkl"
    metadata_path = MODELS_DIR / "anemometer_metadata.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    print("Fitting anemometer model from real data...")
    files = list(Path(real_data_dir).rglob("*.txt"))
    data  = pd.concat([
        pd.read_csv(f, skiprows=6, header=None,
                    usecols=[1], names=['air_flow'])
        for f in files
    ]).reset_index(drop=True)

    metadata = Metadata.detect_from_dataframe(data)
    metadata.save_to_json(str(metadata_path))
    print(f"✓ Metadata saved to {metadata_path}")

    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(data)
    synthesizer.save(str(model_path))
    print(f"✓ Model saved to {model_path}")
    return synthesizer

def load_anemometer():
    model_path = MODELS_DIR / "anemometer_synthesizer.pkl"
    assert model_path.exists(), \
        f"No cached model found at {model_path} — run fit_anemometer() first"
    print(f"Loading cached model from {model_path}")
    return load_synthesizer(str(model_path))

# ── write ─────────────────────────────────────────────────────────────────────
def write_anemometer(values, sensor_id, start, output_dir):
    timestamps = pd.date_range(start=start, periods=len(values), freq='60s')
    max_val    = values.max()
    min_val    = values.min()
    mean_val   = values.mean()
    max_ts     = timestamps[values.argmax()].strftime('%d-%m-%Y,%H:%M:%S')
    min_ts     = timestamps[values.argmin()].strftime('%d-%m-%Y,%H:%M:%S')
    start_str  = timestamps[0].strftime('%d-%m-%Y,%H:%M:%S')

    header = (
        f"TXT Data File\n"
        f"StartTime: {start_str}\n"
        f"Max.: {max_val:.2f} @ {max_ts}  m/s\n"
        f"Min.: {min_val:.2f} @ {min_ts}  m/s\n"
        f"Average: {mean_val:.2f}  m/s\n"
        f"SampleRate: 60 second\n"
        f"\n"
    )
    rows = [
        f"{i}, {val:.2f}, m/s, {ts.strftime('%d-%m-%Y')},{ts.strftime('%H:%M:%S')}"
        for i, (ts, val) in enumerate(zip(timestamps, values), start=1)
    ]

    # match real filename format: CP202526_E111_2022-12-20 9-56.txt
    ts      = pd.Timestamp(start)
    fname   = f"{sensor_id}_{ts.year}-{ts.month:02d}-{ts.day:02d} {ts.hour}-{ts.minute:02d}.txt"
    out_path = Path(output_dir) / fname

    with open(out_path, 'w') as f:
        f.write(header)
        f.write('\n'.join(rows))

    return out_path

# ── edge cases ────────────────────────────────────────────────────────────────
def inject_edge_cases(values, rng):
    start = rng.integers(50, len(values) - 30)
    values[start:start + 20] = 0.0
    neg_idx = rng.choice(len(values), size=2, replace=False)
    values[neg_idx] = -0.1
    return values

# ── main ──────────────────────────────────────────────────────────────────────
def generate_anemometer(real_data_dir, output_dir, sensor_id,
                         start, n_days=7, seed=42, force_refit=False):

    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir)

    model_path = MODELS_DIR / "anemometer_synthesizer.pkl"
    if force_refit or not model_path.exists():
        synthesizer = fit_anemometer(real_data_dir)
    else:
        synthesizer = load_anemometer()

    rng       = np.random.default_rng(seed)
    n_records = n_days * 24 * 60
    synthetic = synthesizer.sample(num_rows=n_records)
    values    = synthetic['air_flow'].clip(lower=0).values
    values    = inject_edge_cases(values, rng)
    out       = write_anemometer(values, sensor_id,
                                 start=pd.Timestamp(start),
                                 output_dir=output_dir)
    print(f"✓ Wrote {out}")
    return out

def generate_anemometer_campaign(real_data_dir, output_dir,
                                  n_sensors=3, n_days=7,
                                  start='2022-09-15 08:00:00',
                                  seed=42, force_refit=False):
    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir)

    model_path = MODELS_DIR / "anemometer_synthesizer.pkl"
    synthesizer = (fit_anemometer(real_data_dir)
                   if force_refit or not model_path.exists()
                   else load_anemometer())

    start     = pd.Timestamp(start)
    n_records = n_days * 24 * 60
    outputs   = []

    for i in range(n_sensors):
        sensor_id = f"CP{i+1:06d}_E{i+1:03d}"
        rng       = np.random.default_rng(seed + i)  # unique seed per sensor

        synthetic = synthesizer.sample(num_rows=n_records)
        values    = synthetic['air_flow'].clip(lower=0).values
        values    = inject_edge_cases(values, rng)

        out = write_anemometer(values, sensor_id, start, output_dir)
        outputs.append(out)
        print(f"✓ Sensor {i+1}/{n_sensors}: {out.name}")

    print(f"\n✓ Generated {n_sensors} sensors × {n_days} days "
          f"({n_records} records each)")
    return outputs

# ── usage ─────────────────────────────────────────────────────────────────────

    generate_anemometer(
        real_data_dir = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/anemometer",
        output_dir    = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer",
        sensor_id     = "CP000001_E001",
        start         = "2022-09-15 08:00:00",
        n_days        = 7,
        seed          = 42
    )

In [12]:
generate_anemometer_campaign(
    real_data_dir = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/anemometer",
    output_dir    = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer",
    n_sensors     = 50,
    n_days        = 100,
    start         = "2022-09-15 08:00:00",
    seed          = 42
)

✓ Found 3 real files in /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/anemometer
✓ Output directory ready: /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer
Loading cached model from synthetic/models/anemometer_synthesizer.pkl
✓ Sensor 1/50: CP000001_E001_2022-09-15 8-00.txt
✓ Sensor 2/50: CP000002_E002_2022-09-15 8-00.txt
✓ Sensor 3/50: CP000003_E003_2022-09-15 8-00.txt
✓ Sensor 4/50: CP000004_E004_2022-09-15 8-00.txt
✓ Sensor 5/50: CP000005_E005_2022-09-15 8-00.txt
✓ Sensor 6/50: CP000006_E006_2022-09-15 8-00.txt
✓ Sensor 7/50: CP000007_E007_2022-09-15 8-00.txt
✓ Sensor 8/50: CP000008_E008_2022-09-15 8-00.txt
✓ Sensor 9/50: CP000009_E009_2022-09-15 8-00.txt
✓ Sensor 10/50: CP000010_E010_2022-09-15 8-00.txt
✓ Sensor 11/50: CP000011_E011_2022-09-15 8-00.txt
✓ Sensor 12/50: CP000012_E012_2022-09-15 8-00.txt
✓ Sensor 13/50: CP000013_E013_2022-09-15 8-00.txt
✓ Sensor 14/50: CP000014_E014_2022-09-15 8-

[PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000001_E001_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000002_E002_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000003_E003_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000004_E004_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000005_E005_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/anemometer/CP000006_E006_2022-09-15 8-00.txt'),
 PosixPath('/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_syntheti

In [16]:
import random
import string

def make_aranet_filename(sensor_id, start):
    """Match exact format: Aranet4 16A28_2022-10-06T15_56_44-0700.csv"""
    ts       = pd.Timestamp(start)
    time_str = ts.strftime('%Y-%m-%dT%H_%M_%S')
    code     = '0700'
    return f"Aranet4 {sensor_id}_{time_str}-{code}.csv"

def fit_aranet(real_data_dir):
    model_path    = MODELS_DIR / "aranet_synthesizer.pkl"
    metadata_path = MODELS_DIR / "aranet_metadata.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    print("Fitting Aranet4 model from real data...")
    files = list(Path(real_data_dir).rglob("*.csv"))
    data  = pd.concat([
        pd.read_csv(f, usecols=[1, 2, 3, 4],
                    names=['co2', 'temperature', 'rh', 'pressure'],
                    skiprows=1)
        for f in files
    ]).reset_index(drop=True)

    # co2 and pressure can have missing first row — drop NaNs
    data = data.dropna()

    metadata = Metadata.detect_from_dataframe(data)
    metadata.save_to_json(str(metadata_path))
    print(f"✓ Metadata saved to {metadata_path}")

    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(data)
    synthesizer.save(str(model_path))
    print(f"✓ Model saved to {model_path}")
    return synthesizer

def load_aranet():
    model_path = MODELS_DIR / "aranet_synthesizer.pkl"
    assert model_path.exists(), \
        f"No cached model found at {model_path} — run fit_aranet() first"
    print(f"Loading cached model from {model_path}")
    return load_synthesizer(str(model_path))

def write_aranet(df, sensor_id, start, output_dir):
    """Write synthetic values to exact raw file format"""
    timestamps = pd.date_range(start=start, periods=len(df), freq='5min')

    # build output dataframe in exact raw format
    out = pd.DataFrame({
        'Time(dd/mm/yyyy)':          timestamps.strftime('%d/%m/%Y %I:%M:%S %p'),
        'Carbon dioxide(ppm)':       df['co2'].round(0).astype('Int64'),
        'Temperature(°C)':           df['temperature'].round(1),
        'Relative humidity(%)':      df['rh'].round(0).astype('Int64'),
        'Atmospheric pressure(hPa)': df['pressure'].round(0).astype('Int64'),
    })

    fname    = make_aranet_filename(sensor_id, start)
    out_path = Path(output_dir) / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_csv(out_path, index=False)
    return out_path

def inject_aranet_edge_cases(df, rng):
    """Inject known Aranet edge cases"""
    # co2 low — sensor startup
    start = rng.integers(0, 5)
    df.loc[start:start+3, 'co2'] = 350.0

    # NaN fill in pressure (first row known issue)
    df.loc[0, 'pressure'] = float('nan')

    # duplicate rows with NaN fills
    dupe_idx = rng.integers(10, len(df) - 10)
    dupe     = df.loc[dupe_idx].copy()
    dupe['co2'] = float('nan')
    df = pd.concat([df.iloc[:dupe_idx+1], pd.DataFrame([dupe]),
                    df.iloc[dupe_idx+1:]]).reset_index(drop=True)
    return df

def generate_aranet_campaign(real_data_dir, output_dir,
                              n_sensors=3, n_days=7,
                              start='2022-09-15 08:00:00',
                              seed=42, force_refit=False):
    """
    Generate synthetic Aranet4 data for multiple sensors.

    Parameters
    ----------
    real_data_dir : str — path to real Aranet4 data for model fitting
    output_dir    : str — path to write synthetic files
    n_sensors     : int — number of synthetic sensors to generate
    n_days        : int — deployment duration per sensor in days
    start         : str — campaign start datetime
    seed          : int — random seed for reproducibility
    force_refit   : bool — refit model even if cached version exists
    """
    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir,
                                               ext='.csv')

    model_path  = MODELS_DIR / "aranet_synthesizer.pkl"
    synthesizer = (fit_aranet(real_data_dir)
                   if force_refit or not model_path.exists()
                   else load_aranet())

    start     = pd.Timestamp(start)
    n_records = n_days * 24 * 12  # 5-min resolution
    outputs   = []

    for i in range(n_sensors):
        # fake sensor IDs — 5 char hex-style like real Aranet IDs
        sensor_id = f"{i+1:02X}{random.randint(0, 0xFFF):03X}"
        rng       = np.random.default_rng(seed + i)

        synthetic = synthesizer.sample(num_rows=n_records)

        # clip to plausible ranges
        synthetic['co2']         = synthetic['co2'].clip(300, 5000).round(0)
        synthetic['temperature'] = synthetic['temperature'].clip(-5, 50).round(1)
        synthetic['rh']          = synthetic['rh'].clip(0, 100).round(0)
        synthetic['pressure']    = synthetic['pressure'].clip(950, 1050).round(0)

        synthetic = inject_aranet_edge_cases(synthetic, rng)

        out = write_aranet(synthetic, sensor_id, start, output_dir)
        outputs.append(out)
        print(f"✓ Sensor {i+1}/{n_sensors}: {out.name}")

    print(f"\n✓ Generated {n_sensors} sensors × {n_days} days "
          f"({n_records} records each)")
    return outputs

# ── usage ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    generate_aranet_campaign(
        real_data_dir = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/aranet",
        output_dir    = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/aranet",
        n_sensors     = 50,
        n_days        = 100,
        start         = "2022-09-15 08:00:00",
        seed          = 42
    )

✓ Found 3 real files in /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/aranet
✓ Output directory ready: /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/aranet
Loading cached model from synthetic/models/aranet_synthesizer.pkl
✓ Sensor 1/50: Aranet4 01DC8_2022-09-15T08_00_00-0700.csv
✓ Sensor 2/50: Aranet4 02984_2022-09-15T08_00_00-0700.csv
✓ Sensor 3/50: Aranet4 03C2F_2022-09-15T08_00_00-0700.csv
✓ Sensor 4/50: Aranet4 04297_2022-09-15T08_00_00-0700.csv
✓ Sensor 5/50: Aranet4 054F4_2022-09-15T08_00_00-0700.csv
✓ Sensor 6/50: Aranet4 06203_2022-09-15T08_00_00-0700.csv
✓ Sensor 7/50: Aranet4 0737D_2022-09-15T08_00_00-0700.csv
✓ Sensor 8/50: Aranet4 081F1_2022-09-15T08_00_00-0700.csv
✓ Sensor 9/50: Aranet4 0990E_2022-09-15T08_00_00-0700.csv
✓ Sensor 10/50: Aranet4 0A87E_2022-09-15T08_00_00-0700.csv
✓ Sensor 11/50: Aranet4 0B127_2022-09-15T08_00_00-0700.csv
✓ Sensor 12/50: Aranet4 0C0AF_2022-09-15T08_00_00-0700.c

In [18]:
def make_lascar_filename(sensor_id, serial, start):
    """Match exact format: 2021_11_18_1000000009_PER_009.txt"""
    ts = pd.Timestamp(start)
    return f"{ts.strftime('%Y_%m_%d')}_{serial}_PER_{sensor_id}.txt"

def fit_lascar(real_data_dir):
    model_path    = MODELS_DIR / "lascar_synthesizer.pkl"
    metadata_path = MODELS_DIR / "lascar_metadata.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    print("Fitting Lascar model from real data...")
    files = list(Path(real_data_dir).rglob("*.txt"))
    data  = pd.concat([
        pd.read_csv(f, usecols=[1, 2], names=['datetime', 'co'],
                    skiprows=1)
        for f in files
    ]).reset_index(drop=True)

    data = data[['co']].dropna()

    metadata = Metadata.detect_from_dataframe(data)
    metadata.save_to_json(str(metadata_path))
    print(f"✓ Metadata saved to {metadata_path}")

    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(data)
    synthesizer.save(str(model_path))
    print(f"✓ Model saved to {model_path}")
    return synthesizer

def load_lascar():
    model_path = MODELS_DIR / "lascar_synthesizer.pkl"
    assert model_path.exists(), \
        f"No cached model found at {model_path} — run fit_lascar() first"
    print(f"Loading cached model from {model_path}")
    return load_synthesizer(str(model_path))

def write_lascar(df, sensor_id, serial, start, output_dir):
    """Write synthetic values to exact raw file format"""
    timestamps = pd.date_range(start=start, periods=len(df), freq='60s')

    # header row matches real format: row count on first line
    n_rows   = len(df)
    header_1 = f"{n_rows:03d},Time,CO(ppm),Serial Number,Sensor Life Expiry,Overrange Exposure"

    rows = [header_1]
    for i, (ts, co) in enumerate(zip(timestamps, df['co']), start=1):
        if i == 1:
            # only first row has serial number and expiry
            rows.append(f"{i},{ts.strftime('%Y-%m-%d %H:%M:%S')},{co:.1f},{serial},2/6/2024,No")
        else:
            rows.append(f"{i},{ts.strftime('%Y-%m-%d %H:%M:%S')},{co:.1f}")

    fname    = make_lascar_filename(sensor_id, serial, start)
    out_path = Path(output_dir) / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, 'w') as f:
        f.write('\n'.join(rows))

    return out_path

def inject_lascar_edge_cases(df, rng):
    """Inject known Lascar edge cases"""
    # negative CO (test flag)
    neg_idx = rng.integers(10, len(df) - 10)
    df.loc[neg_idx, 'co'] = -0.1

    # high CO spike (test flag)
    spike_idx = rng.integers(20, len(df) - 20)
    df.loc[spike_idx, 'co'] = 210.0

    # sustained zeros (filter obstruction)
    zero_start = rng.integers(50, len(df) - 30)
    df.loc[zero_start:zero_start + 15, 'co'] = 0.0

    return df

def generate_lascar_campaign(real_data_dir, output_dir,
                              n_sensors=3, n_days=7,
                              start='2022-09-15 08:00:00',
                              seed=42, force_refit=False):
    """
    Generate synthetic Lascar CO data for multiple sensors.

    Parameters
    ----------
    real_data_dir : str — path to real Lascar data for model fitting
    output_dir    : str — path to write synthetic files
    n_sensors     : int — number of synthetic sensors to generate
    n_days        : int — deployment duration per sensor in days
    start         : str — campaign start datetime
    seed          : int — random seed for reproducibility
    force_refit   : bool — refit model even if cached version exists
    """
    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir,
                                               ext='.txt')

    model_path  = MODELS_DIR / "lascar_synthesizer.pkl"
    synthesizer = (fit_lascar(real_data_dir)
                   if force_refit or not model_path.exists()
                   else load_lascar())

    start     = pd.Timestamp(start)
    n_records = n_days * 24 * 60  # 60s resolution
    outputs   = []

    for i in range(n_sensors):
        sensor_id = f"{i+1:03d}"                        # e.g. 001, 002
        serial    = f"{1000000000 + i + 1:010d}"        # e.g. 1000000001
        rng       = np.random.default_rng(seed + i)

        synthetic = synthesizer.sample(num_rows=n_records)
        synthetic['co'] = synthetic['co'].clip(lower=0.0).round(1)
        synthetic = inject_lascar_edge_cases(synthetic, rng)

        out = write_lascar(synthetic, sensor_id, serial, start, output_dir)
        outputs.append(out)
        print(f"✓ Sensor {i+1}/{n_sensors}: {out.name}")

    print(f"\n✓ Generated {n_sensors} sensors × {n_days} days "
          f"({n_records} records each)")
    return outputs

# ── usage ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    generate_lascar_campaign(
        real_data_dir = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/lascar",
        output_dir    = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/lascar",
        n_sensors     = 50,
        n_days        = 100,
        start         = "2022-09-15 08:00:00",
        seed          = 42
    )

✓ Found 3 real files in /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/lascar
✓ Output directory ready: /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/lascar
Loading cached model from synthetic/models/lascar_synthesizer.pkl
✓ Sensor 1/50: 2022_09_15_1000000001_PER_001.txt
✓ Sensor 2/50: 2022_09_15_1000000002_PER_002.txt
✓ Sensor 3/50: 2022_09_15_1000000003_PER_003.txt
✓ Sensor 4/50: 2022_09_15_1000000004_PER_004.txt
✓ Sensor 5/50: 2022_09_15_1000000005_PER_005.txt
✓ Sensor 6/50: 2022_09_15_1000000006_PER_006.txt
✓ Sensor 7/50: 2022_09_15_1000000007_PER_007.txt
✓ Sensor 8/50: 2022_09_15_1000000008_PER_008.txt
✓ Sensor 9/50: 2022_09_15_1000000009_PER_009.txt
✓ Sensor 10/50: 2022_09_15_1000000010_PER_010.txt
✓ Sensor 11/50: 2022_09_15_1000000011_PER_011.txt
✓ Sensor 12/50: 2022_09_15_1000000012_PER_012.txt
✓ Sensor 13/50: 2022_09_15_1000000013_PER_013.txt
✓ Sensor 14/50: 2022_09_15_1000000014_PER_014.txt
✓ Sen

In [41]:
# ── column definitions ────────────────────────────────────────────────────────
COMMON_COLS = [
    'AccelX', 'AccelY', 'AccelZ', 'Vbus', 'M.Vsupply', 'M.5V0', 'M.3V3',
    'Vbattery', 'Battery_Temp', 'M.BMP581_Press', 'M.BMP581_Temp',
    'SEN55_PM1.0', 'SEN55_PM2.5', 'SEN55_PM4.0', 'SEN55_PM10',
    'SEN55_RH', 'SEN55_Temp', 'SEN55_RawVOC', 'SEN55_RawNOx',
    'G.5V0', 'G.3V3', 'G.BMP581_Press', 'G.BMP581_Temp',
    'G.SCD30_CO2', 'G.SCD30_Temp', 'G.SCD30_RH',
    'G.WE1', 'G.AUX1', 'G.WE2', 'G.AUX2'
]

OLD_FIRMWARE_COLS = [
    'A.Vsupply', 'A.3V3', 'A.BMP581Int_Press', 'A.BMP581Int_Temp',
    'A.BMP581Ext_Press', 'A.BMP581Ext_Temp', 'A.AtmoDensity',
    'G.SGP41_RawVOC', 'G.SGP41_RawNOx',
]

NEW_FIRMWARE_COLS = [
    'A.Vsupply', 'A.3V3', 'A.BMP581Int_Press', 'A.BMP581Int_Temp',
    'A.BMP581Ext_Press', 'A.BMP581Ext_Temp', 'A.AtmoDensity',
    'G.Alphasense1_Algorithm1', 'G.Alphasense1_Algorithm2',
    'G.Alphasense1_Algorithm3', 'G.Alphasense1_Algorithm4',
    'G.Alphasense2_Algorithm1', 'G.Alphasense2_Algorithm2',
    'G.Alphasense2_Algorithm3', 'G.Alphasense2_Algorithm4',
    'D.MassFlow', 'D.VolFlow', 'C.MassFlow', 'C.VolFlow',
    'A.MassFlow', 'A.VolFlow', 'B.MassFlow', 'B.VolFlow',
]

HHB_COLUMNS = {
    'v1': COMMON_COLS + OLD_FIRMWARE_COLS,
    'v2': COMMON_COLS + NEW_FIRMWARE_COLS,
}

HHB_UNITS = {
    'v1': (
        "(HH:MM:SS),(YYYY-MM-DDTHH:MM:SS) (UTC date time format),"
        "(integer),(integer),(integer),"
        "(V),(V),(V),(V),(V),(C),(PaA),(C),"
        "(ug*m^-3),(ug*m^-3),(ug*m^-3),(ug*m^-3),"
        "(%),(C),(integer),(integer),"
        "(V),(V),(PaA),(C),(PaA),(C),(gL^-1),"
        "(V),(V),(PaA),(C),"
        "(ppm),(C),(%),"
        "(integer),(integer),(V),(V),(V),(V)"
    ),
    'v2': (
        "(HH:MM:SS),(YYYY-MM-DDTHH:MM:SS) (UTC date time format),"
        "(integer),(integer),(integer),"
        "(V),(V),(V),(V),(V),(C),(PaA),(C),"
        "(ug*m^-3),(ug*m^-3),(ug*m^-3),(ug*m^-3),"
        "(%),(C),(integer),(integer),"
        "(V),(V),(PaA),(C),(integer),"
        "(V),(V),(V),(PaA),(C),(g*min^-1),(L*min^-1),(L),(L),(L),"
        "(integer),(V),(V),(V),(PaA),(C),(g*min^-1),(L*min^-1),(L),(L),(L),"
        "(gL^-1),(V),(V),(integer),(integer),(V),(V),(V),(PaA),(C),(PaA),(C),"
        "(g*min^-1),(L*min^-1),(L),(L),(L),(gL^-1),"
        "(V),(V),(integer),(integer),(V),(V),(V),(PaA),(C),(PaA),(C),"
        "(g*min^-1),(L*min^-1),(L),(L),(L),(gL^-1),"
        "(V),(V),(PaA),(C),(ppm),(C),(%),"
        "(V),(V),(ppb),(ppb),(ppb),(ppb),"
        "(V),(V),(ppb),(ppb),(ppb),(ppb)"
    ),
}

INT_COLS = {
    'AccelX', 'AccelY', 'AccelZ',
    'SEN55_RawVOC', 'SEN55_RawNOx',
    'G.SGP41_RawVOC', 'G.SGP41_RawNOx',
    'A.Pumps', 'B.Pumps', 'A.RDAC', 'B.RDAC', 'C.RDAC', 'D.RDAC',
    'G.Alphasense1_Algorithm1', 'G.Alphasense1_Algorithm2',
    'G.Alphasense1_Algorithm3', 'G.Alphasense1_Algorithm4',
    'G.Alphasense2_Algorithm1', 'G.Alphasense2_Algorithm2',
    'G.Alphasense2_Algorithm3', 'G.Alphasense2_Algorithm4',
}

# ── helpers ───────────────────────────────────────────────────────────────────
def find_data_start(filepath):
    with open(filepath, 'r') as f:
        for i, line in enumerate(f):
            if line.startswith('SampleTime'):
                return i + 1
    raise ValueError(f"Could not find SampleTime header in {filepath}")

def detect_firmware(f):
    skiprows = find_data_start(f) - 1
    df       = pd.read_csv(f, skiprows=skiprows, nrows=0)
    return 'v1' if len(df.columns) <= 42 else 'v2'

def make_hhb_filename(serial, start):
    ts = pd.Timestamp(start).tz_localize('UTC')
    return f"{serial}_LOG_{ts.strftime('%Y-%m-%dT%H_%M')}UTC.csv"

# ── header ────────────────────────────────────────────────────────────────────
def make_hhb_header(serial, fname, start, n_days, utc_offset=-4.0,
                    alphasense1_id='0000000001', alphasense2_id='0000000002',
                    firmware='v1'):
    ts_start  = pd.Timestamp(start).tz_localize('UTC')
    ts_end    = ts_start + pd.Timedelta(days=n_days)
    runtime   = n_days * 24
    col_names = ','.join(['SampleTime', 'DateTimeUTC'] + HHB_COLUMNS[firmware])
    units_row = HHB_UNITS[firmware]

    return (
        f"PARAMETER,VALUE,UNITS/NOTES\n"
        f"\n"
        f"HHBserial,{serial},(HHB box serial identification)\n"
        f"SEN55_Serial,SYNTHETIC000000000,(SEN55 serial identification)\n"
        f"HHBslot1,Empty,(No expansion board)\n"
        f"HHBslot2,FP00001,(A Filter pumping expansion board)\n"
        f"HHBslot3,Empty,(No expansion board)\n"
        f"HHBslot4,Empty,(No expansion board)\n"
        f"HHBslot5,Empty,(No expansion board)\n"
        f"HHBslot6,GS00001,(Gas sensor expansion board)\n"
        f"G.Alphasense1_ID,{alphasense1_id},(Alphasense #1 on gas expansion board)\n"
        f"G.Alphasense2_ID,{alphasense2_id},(Alphasense #2 on gas expansion board)\n"
        f"G.SCD30_Serial,SYNTHETIC-SCD30,(SCD30 on gas expansion board)\n"
        f"G.SGP41_Serial,SYNTHETIC41,(SGP41 on gas expansion board)\n"
        f"Firmware,HHBv2 Jan 11 2024 08:35:16,(installed firmware version)\n"
        f"\n\n\n"
        f"\n"
        f"SAMPLE IDENTIFICATION\n"
        f"\n"
        f"LogFileName,{fname},(log file filename-automatically defined)\n"
        f"SampleName,SYNTHETIC,(sample name)\n"
        f"A.FilterCID,,(A Filter cartridge ID)\n"
        f"\n\n\n"
        f"\n"
        f"MASS FLOW SENSOR CALIBRATION\n"
        f"\n"
        f"A.FilterCalDate,2024-05-16T10:00:00,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"A.FilterCalVoutMin,0.595875,(V)\n"
        f"A.FilterCalVoutMax,1.891500,(V)\n"
        f"A.FilterCalMFMin,0.251100,(g*min^-1)\n"
        f"A.FilterCalMFMax,3.473000,(g*min^-1)\n"
        f"A.FilterMF4,0.685189,(coefficient)\n"
        f"A.FilterMF3,-1.948710,(coefficient)\n"
        f"A.FilterMF2,1.932300,(coefficient)\n"
        f"A.FilterMF1,0.826531,(coefficient)\n"
        f"A.FilterMF0,-0.601246,(coefficient)\n"
        f"\n\n\n"
        f"\n"
        f"SETUP SUMMARY\n"
        f"\n"
        f"UTCOffset,{utc_offset:.2f},(hours offset from UTC date time)\n"
        f"ProgrammedRuntime,{n_days * 86400:.6f},(s)\n"
        f"SEN55_Runtime,{runtime:.3f},(Hr)\n"
        f"SEN55_FanRuntime,{runtime:.3f},(Hr)\n"
        f"A.FilterPumpStartingVolume,8.64,(L)\n"
        f"A.FilterPumpStartingRuntime,0.100,(Hr)\n"
        f"A.FilterCartridgeStartingVolume,8.64,(L)\n"
        f"A.FilterCartridgeStartingRuntime,0.000,(Hr)\n"
        f"A.FilterVolumetricFlowRate,0.00,(L*min^-1)\n"
        f"A.FilterDutyCycle,100.0,(%)\n"
        f"G.Alphasense1_Runtime,{runtime:.3f},(Hr)\n"
        f"G.Alphasense2_Runtime,{runtime:.3f},(Hr)\n"
        f"G.FanRuntime,{runtime:.3f},(Hr)\n"
        f"G.SCD30_Runtime,{runtime:.3f},(Hr)\n"
        f"\n\n\n"
        f"\n"
        f"SAMPLE SUMMARY\n"
        f"\n"
        f"StartDateTimeUTC,{ts_start.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"EndDateTimeUTC,{ts_end.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"HHBSampledRuntime,        {runtime:.3f},(Hr)\n"
        f"A.FilterShutdownMode,0,(0=uncontrolled 1=test finished 2=high power)\n"
        f"A.FilterSampledRunTime,          0.000,(Hr)\n"
        f"A.FilterSampledVolume,           0.00,(L)\n"
        f"A.FilterAverageVolumetricFlowRate,          0.000,(L*min^-1)\n"
        f"\n\n\n"
        f"\n"
        f"SAMPLE LOG\n"
        f"\n"
        f"{units_row}\n"
        f"{col_names}"
    )

# ── fit / load ────────────────────────────────────────────────────────────────
def fit_hhb(real_data_dir):
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    files = list(Path(real_data_dir).rglob("HHB*.csv"))
    assert len(files) > 0, f"No HHB*.csv files found in {real_data_dir}"

    v1_files = [f for f in files if detect_firmware(f) == 'v1']
    v2_files = [f for f in files if detect_firmware(f) == 'v2']
    print(f"Firmware v1: {len(v1_files)} files, v2: {len(v2_files)} files")

    models = {}
    for version, flist, extra_cols in [
        ('v1', v1_files, OLD_FIRMWARE_COLS),
        ('v2', v2_files, NEW_FIRMWARE_COLS),
    ]:
        if not flist:
            print(f"No {version} files — skipping")
            continue

        cols = COMMON_COLS + extra_cols
        dfs  = []
        for f in flist:
            try:
                skiprows  = find_data_start(f) - 1
                df        = pd.read_csv(f, skiprows=skiprows)
                available = [c for c in cols if c in df.columns]
                dfs.append(df[available])
                print(f"  ✓ {f.name} ({version}): {len(df)} rows")
            except Exception as e:
                print(f"  ✗ {f.name}: {e} — skipping")

        if not dfs:
            print(f"  No valid {version} files — skipping model")
            continue

        data          = pd.concat(dfs).reset_index(drop=True).dropna()
        metadata      = Metadata.detect_from_dataframe(data)
        model_path    = MODELS_DIR / f"hhb_{version}_synthesizer.pkl"
        metadata_path = MODELS_DIR / f"hhb_{version}_metadata.json"
        metadata.save_to_json(str(metadata_path))

        synthesizer = GaussianCopulaSynthesizer(metadata)
        synthesizer.fit(data)
        synthesizer.save(str(model_path))
        models[version] = synthesizer
        print(f"✓ HHB {version} model saved to {model_path}")

    return models

def load_hhb(firmware='v1'):
    model_path = MODELS_DIR / f"hhb_{firmware}_synthesizer.pkl"
    assert model_path.exists(), \
        f"No cached {firmware} model — run fit_hhb() first"
    print(f"Loading cached {firmware} model from {model_path}")
    return load_synthesizer(str(model_path))

# ── write ─────────────────────────────────────────────────────────────────────
def write_hhb(df, serial, start, n_days, output_dir,
              utc_offset=-4.0, alphasense1_id='0000000001',
              alphasense2_id='0000000002', firmware='v1'):
    ts_start   = pd.Timestamp(start).tz_localize('UTC')
    timestamps = pd.date_range(start=ts_start, periods=len(df), freq='30s')
    cols       = HHB_COLUMNS[firmware]
    fname      = make_hhb_filename(serial, start)
    header     = make_hhb_header(serial, fname, start, n_days,
                                  utc_offset, alphasense1_id,
                                  alphasense2_id, firmware)
    rows = []
    for i, (ts, (_, row)) in enumerate(zip(timestamps, df.iterrows())):
        elapsed     = pd.Timedelta(seconds=i * 30)
        h, rem      = divmod(int(elapsed.total_seconds()), 3600)
        m, s        = divmod(rem, 60)
        sample_time = f"{h}:{m:02d}:{s:02d}"
        dt_utc      = ts.strftime('%Y-%m-%dT%H:%M:%S')
        values      = []
        for col in cols:
            if col not in row.index:
                values.append('0')
            elif col in INT_COLS:
                values.append(str(int(row[col])))
            else:
                values.append(f"{row[col]:.3f}")
        rows.append(f"{sample_time},{dt_utc},{','.join(values)}")

    out_path = Path(output_dir) / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        f.write(header)
        f.write('\n')
        f.write('\n'.join(rows))
    return out_path

# ── edge cases ────────────────────────────────────────────────────────────────
def inject_hhb_edge_cases(df, rng):
    neg_idx    = rng.integers(10, len(df) - 10)
    batt_start = rng.integers(100, len(df) - 50)
    df.loc[neg_idx, 'SEN55_PM2.5']          = -0.1
    df.loc[batt_start:batt_start+20, 'Vbattery'] = 3.1
    df.loc[0:10, 'G.SCD30_CO2']             = 1200.0
    return df

# ── campaign ──────────────────────────────────────────────────────────────────
def generate_hhb_campaign(real_data_dir, output_dir,
                           n_sensors=3, n_days=7,
                           start='2022-09-15 08:00:00',
                           utc_offset=-7.0, firmware='v1',
                           seed=42, force_refit=False):
    """
    Generate synthetic HHB data for multiple sensors.

    Parameters
    ----------
    real_data_dir : str   — path to real HHB raw data for model fitting
    output_dir    : str   — path to write synthetic files
    n_sensors     : int   — number of synthetic sensors to generate
    n_days        : int   — deployment duration per sensor in days
    start         : str   — campaign start datetime (UTC)
    utc_offset    : float — UTC offset for local time in header
    firmware      : str   — 'v1' (older) or 'v2' (newer) firmware format
    seed          : int   — random seed for reproducibility
    force_refit   : bool  — refit model even if cached version exists
    """
    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir,
                                               ext='.csv')
    model_path  = MODELS_DIR / f"hhb_{firmware}_synthesizer.pkl"
    synthesizer = (fit_hhb(real_data_dir)[firmware]
                   if force_refit or not model_path.exists()
                   else load_hhb(firmware))

    n_records = n_days * 24 * 120
    outputs   = []

    for i in range(n_sensors):
        serial    = f"HHB{99000 + i + 1:05d}"
        rng       = np.random.default_rng(seed + i)
        synthetic = synthesizer.sample(num_rows=n_records)

        synthetic['SEN55_PM1.0']  = synthetic['SEN55_PM1.0'].clip(0, 500)
        synthetic['SEN55_PM2.5']  = synthetic['SEN55_PM2.5'].clip(0, 500)
        synthetic['SEN55_PM4.0']  = synthetic['SEN55_PM4.0'].clip(0, 500)
        synthetic['SEN55_PM10']   = synthetic['SEN55_PM10'].clip(0, 500)
        synthetic['SEN55_RH']     = synthetic['SEN55_RH'].clip(0, 100)
        synthetic['SEN55_Temp']   = synthetic['SEN55_Temp'].clip(-5, 50)
        synthetic['G.SCD30_CO2']  = synthetic['G.SCD30_CO2'].clip(400, 5000)
        synthetic['G.SCD30_RH']   = synthetic['G.SCD30_RH'].clip(0, 100)
        synthetic['G.SCD30_Temp'] = synthetic['G.SCD30_Temp'].clip(-5, 50)
        synthetic = inject_hhb_edge_cases(synthetic, rng)

        out = write_hhb(synthetic, serial, start, n_days,
                        output_dir, utc_offset=utc_offset, firmware=firmware)
        outputs.append(out)
        print(f"✓ Sensor {i+1}/{n_sensors}: {out.name}")

    print(f"\n✓ Generated {n_sensors} × {firmware} sensors, "
          f"{n_days} days ({n_records} records each)")
    return outputs

# ── usage ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    generate_hhb_campaign(
        real_data_dir = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/hhb",
        output_dir    = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/hhb",
        n_sensors     = 50,
        n_days        = 100,
        start         = "2022-09-15 08:00:00",
        utc_offset    = -7.0,
        firmware      = 'v1',
        seed          = 42
    )

✓ Found 4 real files in /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/hhb
✓ Output directory ready: /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/hhb
Loading cached v1 model from synthetic/models/hhb_v1_synthesizer.pkl
✓ Sensor 1/50: HHB99001_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 2/50: HHB99002_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 3/50: HHB99003_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 4/50: HHB99004_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 5/50: HHB99005_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 6/50: HHB99006_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 7/50: HHB99007_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 8/50: HHB99008_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 9/50: HHB99009_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 10/50: HHB99010_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 11/50: HHB99011_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 12/50: HHB99012_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 13/50: HHB99013_LOG_2022-09-15T08_00UTC.csv
✓ Sensor 14/50: HHB

In [44]:
# check column count and header length per file
upas_files = list(Path("/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/upas").rglob("PS*.txt"))

for f in upas_files:
    start = find_data_start(f)
    df    = pd.read_csv(f, skiprows=start-1, nrows=0)
    print(f"\n{f.name}")
    print(f"  Header ends at line: {start-1}")
    print(f"  Column count: {len(df.columns)}")
    print(f"  First 5 cols: {list(df.columns)[:5]}")


PSP00054_LOG_2022-09-17T21_59_27UTC_E106____________C3805_____.txt
  Header ends at line: 101
  Column count: 112
  First 5 cols: ['SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal']

PSP00054_LOG_2022-09-06T19_37_17UTC_E102____________C3798_____.txt
  Header ends at line: 101
  Column count: 112
  First 5 cols: ['SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal']

PSP00054_LOG_2022-09-13T18_57_39UTC_E103____________C3858_____.txt
  Header ends at line: 101
  Column count: 112
  First 5 cols: ['SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal']


In [45]:
upas_files = list(Path("/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/upas").rglob("PS*.txt"))

for f in upas_files:
    try:
        start = find_data_start(f)
        df    = pd.read_csv(f, skiprows=start-1, nrows=0)
        print(f"\n{f.name}")
        print(f"  Header ends at line: {start-1}")
        print(f"  Column count: {len(df.columns)}")
        print(f"  Columns: {list(df.columns)}")
    except Exception as e:
        print(f"\n{f.name} — ERROR: {e}")


PSP00054_LOG_2022-09-17T21_59_27UTC_E106____________C3805_____.txt
  Header ends at line: 101
  Column count: 112
  Columns: ['SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal', 'PumpingFlowRate', 'OverallFlowRate', 'SampledVolume', 'FilterDP', 'BatteryCharge', 'AtmoT', 'AtmoP', 'AtmoRH', 'AtmoDensity', 'AtmoAlt', 'GPSQual', 'GPSlat', 'GPSlon', 'GPSalt', 'GPSsat', 'GPSspeed', 'GPShDOP', 'AccelX', 'AccelXVar', 'AccelXMin', 'AccelXMax', 'AccelY', 'AccelYVar', 'AccelYMin', 'AccelYMax', 'AccelZ', 'AccelZVar', 'AccelZMin', 'AccelZMax', 'RotX', 'RotXVar', 'RotXMin', 'RotXMax', 'RotY', 'RotYVar', 'RotYMin', 'RotYMax', 'RotZ', 'RotZVar', 'RotZMin', 'RotZMax', 'Xup', 'XDown', 'Yup', 'Ydown', 'Zup', 'Zdown', 'StepCount', 'LUX', 'UVindex', 'HighVisRaw', 'LowVisRaw', 'IRRaw', 'UVRaw', 'PMMeasCnt', 'PM1MC', 'PM1MCVar', 'PM2_5MC', 'PM2_5MCVar', 'PM4MC', 'PM4MCVar', 'PM10MC', 'PM10MCVar', 'PM0_5NC', 'PM0_5NCVar', 'PM1NC', 'PM1NCVar', 'PM2_5NC', 'PM2_5NCVar', 'PM4NC', 'PM4NCVar',

In [46]:
# peek at one file around the header/data boundary
f = upas_files[0]
with open(f) as fh:
    lines = fh.readlines()
for i, line in enumerate(lines):
    if i > len(lines) - 10:
        break
    if any(x in line for x in ['DateTime', 'SampleTime', 'SAMPLE LOG', 'Date']):
        print(f"{i}: {repr(line[:80])}")

47: 'StartDateTimeUTC,2022-09-17T21:59:27,(YYYY-MM-DDTHH:MM:SS) (UTC date time format'
48: 'StartDateTimeLocal,2022-09-17T14:59:27,(YYYY-MM-DDTHH:MM:SS) (Local date time fo'
49: 'EndDateTimeUTC,2022-09-19T23:41:42,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n'
50: 'EndDateTimeLocal,2022-09-19T16:41:42,(YYYY-MM-DDTHH:MM:SS) (Local date time form'
69: 'CO2CalDate,2022-05-19T19:47:17,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n'
78: 'MFSCalDate,2022-04-18T19:12:00,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n'
98: 'SAMPLE LOG\n'
100: 'DateTime,DateTime,DateTime,DateTime,DateTime,FilterSample,FilterSample,FilterSam'
101: 'SampleTime,UnixTime,UnixTimeMCU,DateTimeUTC,DateTimeLocal,PumpingFlowRate,Overal'


In [47]:
f = upas_files[0]
with open(f) as fh:
    lines = fh.readlines()
for i, line in enumerate(lines[:102]):
    print(f"{i:3d}: {repr(line.rstrip()[:80])}")

  0: 'PARAMETER,VALUE,UNITS/NOTES'
  1: ''
  2: 'UPASserial,00054,(UPAS serial identification-numerical)'
  3: 'UPASpcbRev,0,(UPAS pcb revision number)'
  4: 'UPASexpRev,1,(UPAS expansion pcb)'
  5: 'PMserial,9553DB1A763FBA22_2.2_7_2.0,(SPS30 serial identification_FWver_HWrev_SHD'
  6: 'UPASfirmware,UPAS_v2_x-rev_00127-L476RE_20220603_mbedOS-6_15_1.bin compiled Jun '
  7: 'LifetimeSampleCount,4,(count-total lifetime sample runs)'
  8: 'LifetimeSampleRuntime,143.53,(hrs-total lifetime cumulative sample runtime)'
  9: ''
 10: ''
 11: ''
 12: ''
 13: 'SAMPLE IDENTIFICATION'
 14: ''
 15: 'LogFilename,/sd/20220917/PSP00054_LOG_2022-09-17T21_59_27UTC_E106____________C38'
 16: 'SampleName,E106___________,(Sample Name-user entered into app)'
 17: 'CartridgeID,C3805_____,(Cartridge Identification-user entered into app)'
 18: ''
 19: ''
 20: ''
 21: ''
 22: 'SETUP SUMMARY'
 23: ''
 24: 'GPSUTCOffset,-7.00,(hours offset from UTC date time)'
 25: 'StartOnNextPowerUp,0,(0=no 1=yes 2=system reset)'


In [66]:
# ── column definitions ────────────────────────────────────────────────────────
UPAS_COLS = [
    'SampleTime', 'UnixTime', 'UnixTimeMCU', 'DateTimeUTC', 'DateTimeLocal',
    'PumpingFlowRate', 'OverallFlowRate', 'SampledVolume', 'FilterDP',
    'BatteryCharge', 'AtmoT', 'AtmoP', 'AtmoRH', 'AtmoDensity', 'AtmoAlt',
    'GPSQual', 'GPSlat', 'GPSlon', 'GPSalt', 'GPSsat', 'GPSspeed', 'GPShDOP',
    'AccelX', 'AccelXVar', 'AccelXMin', 'AccelXMax',
    'AccelY', 'AccelYVar', 'AccelYMin', 'AccelYMax',
    'AccelZ', 'AccelZVar', 'AccelZMin', 'AccelZMax',
    'RotX', 'RotXVar', 'RotXMin', 'RotXMax',
    'RotY', 'RotYVar', 'RotYMin', 'RotYMax',
    'RotZ', 'RotZVar', 'RotZMin', 'RotZMax',
    'Xup', 'XDown', 'Yup', 'Ydown', 'Zup', 'Zdown',
    'StepCount', 'LUX', 'UVindex',
    'HighVisRaw', 'LowVisRaw', 'IRRaw', 'UVRaw',
    'PMMeasCnt',
    'PM1MC', 'PM1MCVar', 'PM2_5MC', 'PM2_5MCVar',
    'PM4MC', 'PM4MCVar', 'PM10MC', 'PM10MCVar',
    'PM0_5NC', 'PM0_5NCVar', 'PM1NC', 'PM1NCVar',
    'PM2_5NC', 'PM2_5NCVar', 'PM4NC', 'PM4NCVar',
    'PM10NC', 'PM10NCVar',
    'PMtypicalParticleSize', 'PMtypicalParticleSizeVar',
    'PM2_5SampledMass',
    'PCB1T', 'PCB2T', 'FdpT', 'AccelT', 'PT100R', 'PCB2P',
    'PumpPow1', 'PumpPow2', 'PumpV',
    'MassFlow', 'MFSVout', 'BFGenergy', 'BattVolt',
    'v3_3', 'v5', 'PumpsON', 'Dead',
    'BCS1', 'BCS2', 'BC_NPG', 'FLOWCTL', 'GPSRT',
    'SD_DATAW', 'SD_HEADW', 'TPumpsOFF', 'TPumpsON',
    'CO2', 'SCDT', 'SCDRH', 'VOCRaw', 'NOXRaw'
]

# columns to fit SDV on — exclude datetime, GPS, and engineering columns
UPAS_FIT_COLS = [
    'PumpingFlowRate', 'OverallFlowRate', 'FilterDP', 'BatteryCharge',
    'AtmoT', 'AtmoP', 'AtmoRH', 'AtmoDensity', 'AtmoAlt',
    'AccelX', 'AccelY', 'AccelZ', 'StepCount',
    'LUX', 'UVindex', 'HighVisRaw', 'LowVisRaw', 'IRRaw', 'UVRaw',
    'PMMeasCnt',
    'PM1MC', 'PM1MCVar', 'PM2_5MC', 'PM2_5MCVar',
    'PM4MC', 'PM4MCVar', 'PM10MC', 'PM10MCVar',
    'PM0_5NC', 'PM0_5NCVar', 'PM1NC', 'PM1NCVar',
    'PM2_5NC', 'PM2_5NCVar', 'PM4NC', 'PM4NCVar',
    'PM10NC', 'PM10NCVar',
    'PMtypicalParticleSize', 'PMtypicalParticleSizeVar',
    'PCB1T', 'PCB2T', 'FdpT', 'AccelT', 'MassFlow', 'BattVolt',
    'CO2', 'SCDT', 'SCDRH', 'VOCRaw', 'NOXRaw'
]

UPAS_INT_COLS = {
    'BatteryCharge', 'GPSQual', 'GPSsat', 'PMMeasCnt',
    'AccelX', 'AccelY', 'AccelZ',
    'Xup', 'XDown', 'Yup', 'Ydown', 'Zup', 'Zdown',
    'StepCount', 'PumpsON', 'Dead',
    'BCS1', 'BCS2', 'BC_NPG', 'FLOWCTL', 'GPSRT',
    'SD_DATAW', 'SD_HEADW',
}

# ── filename ──────────────────────────────────────────────────────────────────
def make_upas_filename(serial, start, sample_name='SYNTHETIC',
                        cartridge_id='C9999'):
    """PSP00054_LOG_2022-09-06T19_37_17UTC_E102____________C3798_____.txt"""
    ts        = pd.Timestamp(start).tz_localize('UTC')
    ts_str    = ts.strftime('%Y-%m-%dT%H_%M_%SZ')
    # pad sample name and cartridge to match fixed-width format
    sname_pad = f"{sample_name:<12}"[:12]
    cart_pad  = f"{cartridge_id:<10}"[:10]
    return f"PSP{serial}_LOG_{ts_str}UTC_{sname_pad}_{cart_pad}_.txt"

# ── header ────────────────────────────────────────────────────────────────────
def make_upas_header(serial, fname, start, n_days,
                      sample_name='SYNTHETIC', cartridge_id='C9999',
                      gps_utc_offset=-7.0,
                      co2_cal_target=417, co2_cal_offset=12):
    ts_start   = pd.Timestamp(start).tz_localize('UTC')
    ts_end     = ts_start + pd.Timedelta(days=n_days)
    ts_local   = ts_start + pd.Timedelta(hours=gps_utc_offset)
    te_local   = ts_end   + pd.Timedelta(hours=gps_utc_offset)
    duration   = n_days * 24
    volume     = duration * 60 * 1.0  # at 1 L/min

    # units row — line 100 in real file
    units_row = (
        'DateTime,DateTime,DateTime,DateTime,DateTime,'
        'FilterSample,FilterSample,FilterSample,FilterSample,FilterSample,'
        'Atmospheric,Atmospheric,Atmospheric,Atmospheric,Atmospheric,'
        'GPS,GPS,GPS,GPS,GPS,GPS,GPS,'
        'Accel,Accel,Accel,Accel,Accel,Accel,Accel,Accel,'
        'Accel,Accel,Accel,Accel,Accel,Accel,Accel,Accel,'
        'Accel,Accel,Accel,Accel,Accel,Accel,Accel,Accel,'
        'Orientation,Orientation,Orientation,Orientation,Orientation,Orientation,'
        'Accel,Light,Light,Light,Light,Light,Light,'
        'OPC,OPC,OPC,OPC,OPC,OPC,OPC,OPC,OPC,'
        'OPC,OPC,OPC,OPC,OPC,OPC,OPC,OPC,OPC,'
        'OPC,OPC,OPC,OPC,OPC,OPC,OPC,'
        'PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,PCB,'
        'Status,Status,Status,Status,Status,Status,Status,Status,Status,'
        'CO2,CO2,CO2,CO2,CO2'
    )
    col_row = ','.join(UPAS_COLS)

    return (
        f"PARAMETER,VALUE,UNITS/NOTES\n"
        f"\n"
        f"UPASserial,{serial},(UPAS serial identification-numerical)\n"
        f"UPASpcbRev,0,(UPAS pcb revision number)\n"
        f"UPASexpRev,1,(UPAS expansion pcb)\n"
        f"PMserial,SYNTHETIC_PM,(SPS30 serial identification)\n"
        f"UPASfirmware,UPAS_v2_x-rev_00127-L476RE_20220603_mbedOS-6_15_1.bin compiled Jun  3 2022\n"
        f"LifetimeSampleCount,1,(count-total lifetime sample runs)\n"
        f"LifetimeSampleRuntime,{duration:.2f},(hrs-total lifetime cumulative sample runtime)\n"
        f"\n\n\n"
        f"\n"
        f"SAMPLE IDENTIFICATION\n"
        f"\n"
        f"LogFilename,/sd/{ts_start.strftime('%Y%m%d')}/{fname},(log filename)\n"
        f"SampleName,{sample_name:<11},(Sample Name-user entered into app)\n"
        f"CartridgeID,{cartridge_id:<10},(Cartridge Identification-user entered into app)\n"
        f"\n\n\n"
        f"\n"
        f"SETUP SUMMARY\n"
        f"\n"
        f"GPSUTCOffset,{gps_utc_offset:.2f},(hours offset from UTC date time)\n"
        f"StartOnNextPowerUp,0,(0=no 1=yes 2=system reset)\n"
        f"ProgrammedStartTime,0,(0 = Now or Start On Next or seconds since 1/1/1970)\n"
        f"ProgrammedRuntime,indefinite,(Hr)\n"
        f"SizeSelectiveInlet,PM2.5,(inlet particle size fraction)\n"
        f"FlowRateSetpoint,1.000,(L*min^-1)\n"
        f"FlowOffset,0.000000,(%)\n"
        f"FlowDutyCycle,100,(%)\n"
        f"DutyCycleWindow,30,(s)\n"
        f"GPSEnabled,0,(0=no 1=yes)\n"
        f"PMSensorInterval,1,(0=sensor disabled 1=continuous measurement)\n"
        f"RTGasSampleState,0,(0=off 1=on)\n"
        f"CO2SampleState,1,(0=off 1=on)\n"
        f"LogInterval,30,(s)\n"
        f"PowerSaveMode,0,(0=off 1=on)\n"
        f"AppLock,0,(0=unlocked 1=locked -1=not set)\n"
        f"AppVersion,i1.0.2,(i=iOS A=Android)\n"
        f"\n\n\n"
        f"\n"
        f"SAMPLE SUMMARY\n"
        f"\n"
        f"StartDateTimeUTC,{ts_start.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"StartDateTimeLocal,{ts_local.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (Local date time format)\n"
        f"EndDateTimeUTC,{ts_end.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"EndDateTimeLocal,{te_local.strftime('%Y-%m-%dT%H:%M:%S')},(YYYY-MM-DDTHH:MM:SS) (Local date time format)\n"
        f"FlowCheckMeterReadingPreSample,-99.900,(L*min^-1)\n"
        f"FlowCheckMeterReadingPostSample,NA,(L*min^-1)\n"
        f"OverallDuration, {duration:.3f},(Hr)\n"
        f"PumpingDuration, {duration:.3f},(Hr)\n"
        f"OverallFlowRateAverage,1.000,(L*min^-1)\n"
        f"PumpingFlowRateAverage,1.000,(L*min^-1)\n"
        f"SampledVolume, {volume:.2f},(L)\n"
        f"StartBatteryCharge,099,(%)\n"
        f"EndBatteryCharge,099,(%)\n"
        f"StartBatteryVoltage,4.14,(V)\n"
        f"EndBatteryVoltage,4.10,(V)\n"
        f"ShutdownMode,01,(0=unknown error shutdown 1=user pushbutton sample stop)\n"
        f"\n\n\n"
        f"\n"
        f"CO2 SENSOR CALIBRATION\n"
        f"\n"
        f"CO2CalDate,2022-05-19T19:47:17,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"CO2CalTarget,{co2_cal_target},(ppm)\n"
        f"CO2CalOffset,{co2_cal_offset},(ppm)\n"
        f"\n\n\n"
        f"\n"
        f"MASS FLOW SENSOR CALIBRATION\n"
        f"\n"
        f"MFSCalDate,2022-04-18T19:12:00,(YYYY-MM-DDTHH:MM:SS) (UTC date time format)\n"
        f"MFSCalPerson,,(name of person running and approving calibration)\n"
        f"MFSCalVoutBlocked,,(V)\n"
        f"MFSCalVoutMin,0.480000,(V)\n"
        f"MFSCalVoutMax,2.084125,(V)\n"
        f"MFSCalMFBlocked,,(g*min^-1)\n"
        f"MFSCalMFMin,0.010983,(g*min^-1)\n"
        f"MFSCalMFMax,3.387900,(g*min^-1)\n"
        f"MFSCalPumpVBoostMin,,(V)\n"
        f"MFSCalPumpVBoostMax,,(V)\n"
        f"MFSCalPDeadhead,,(Pa)\n"
        f"MF4,0.134946,(coefficient)\n"
        f"MF3,0.131147,(coefficient)\n"
        f"MF2,-0.948272,(coefficient)\n"
        f"MF1,2.215893,(coefficient)\n"
        f"MF0,-0.855831,(coefficient)\n"
        f"\n\n\n"
        f"SAMPLE LOG\n"
        f"\n"
        f"{units_row}\n"
        f"{col_row}"
    )

# ── fit / load ────────────────────────────────────────────────────────────────
def fit_upas(real_data_dir):
    model_path    = MODELS_DIR / "upas_synthesizer.pkl"
    metadata_path = MODELS_DIR / "upas_metadata.json"
    MODELS_DIR.mkdir(parents=True, exist_ok=True)

    print("Fitting UPAS model from real data...")
    files = list(Path(real_data_dir).rglob("PS*.txt"))
    assert len(files) > 0, f"No PS*.txt files found in {real_data_dir}"

    dfs = []
    for f in files:
        try:
            skiprows  = find_data_start(f) - 1
            df        = pd.read_csv(f, skiprows=skiprows)
            available = [c for c in UPAS_FIT_COLS if c in df.columns]
            df        = df[available]
            # force numeric — some columns read as object/string
            df        = df.apply(pd.to_numeric, errors='coerce').dropna()
            dfs.append(df)
            print(f"  ✓ {f.name}: {len(df)} rows")
        except Exception as e:
            print(f"  ✗ {f.name}: {e} — skipping")

    data     = pd.concat(dfs).reset_index(drop=True).dropna()
    metadata = Metadata.detect_from_dataframe(data)
    metadata.save_to_json(str(metadata_path))
    synthesizer = GaussianCopulaSynthesizer(metadata)
    synthesizer.fit(data)
    synthesizer.save(str(model_path))
    print(f"✓ Model saved to {model_path}")
    return synthesizer

def load_upas():
    model_path = MODELS_DIR / "upas_synthesizer.pkl"
    assert model_path.exists(), \
        f"No cached model — run fit_upas() first"
    print(f"Loading cached UPAS model from {model_path}")
    return load_synthesizer(str(model_path))

# ── write ─────────────────────────────────────────────────────────────────────
def write_upas(df, serial, start, n_days, output_dir,
               sample_name='SYNTHETIC', cartridge_id='C9999',
               gps_utc_offset=-7.0):
    ts_start   = pd.Timestamp(start).tz_localize('UTC')
    ts_local   = ts_start + pd.Timedelta(hours=gps_utc_offset)
    timestamps = pd.date_range(start=ts_start, periods=len(df), freq='30s')
    unix_times = timestamps.astype(np.int64) // 10**9

    fname  = make_upas_filename(serial, start, sample_name, cartridge_id)
    header = make_upas_header(serial, fname, start, n_days,
                               sample_name, cartridge_id, gps_utc_offset)

    # cumulative sampled volume — increases monotonically
    sampled_vol = np.cumsum(np.ones(len(df)) * (1.0 / 120))  # 1 L/min at 30s

    rows = []
    for i, (ts, unix, (_, row)) in enumerate(
            zip(timestamps, unix_times, df.iterrows())):

        elapsed   = pd.Timedelta(seconds=i * 30)
        h, rem    = divmod(int(elapsed.total_seconds()), 3600)
        m, s      = divmod(rem, 60)
        ts_loc    = ts + pd.Timedelta(hours=gps_utc_offset)
        sample_t  = f"{h}:{m:02d}:{s:02d}"
        dt_utc    = ts.strftime('%Y-%m-%dT%H:%M:%S')
        dt_local  = ts_loc.strftime('%Y-%m-%dT%H:%M:%S')

        def fmt(col, default=0.0):
            if col not in row.index:
                return str(int(default)) if col in UPAS_INT_COLS else f"{default:.3f}"
            val = row[col]
            return str(int(val)) if col in UPAS_INT_COLS else f"{val:.3f}"

        values = [
            sample_t,
            str(unix),
            str(unix),                      # UnixTimeMCU ~ UnixTime
            dt_utc,
            dt_local,
            fmt('PumpingFlowRate', 1.0),
            fmt('OverallFlowRate', 1.0),
            f"{sampled_vol[i]:.2f}",        # cumulative
            fmt('FilterDP'),
            fmt('BatteryCharge', 99),
            fmt('AtmoT'), fmt('AtmoP'), fmt('AtmoRH'),
            fmt('AtmoDensity'), fmt('AtmoAlt'),
            '0', '-999.0', '-999.0', '-999.0', '0', '0.0', '99.0',  # GPS (disabled)
            fmt('AccelX'), '0.0', '0.0', '0.0',   # AccelX + Var/Min/Max
            fmt('AccelY'), '0.0', '0.0', '0.0',
            fmt('AccelZ'), '0.0', '0.0', '0.0',
            '0.0', '0.0', '0.0', '0.0',           # RotX
            '0.0', '0.0', '0.0', '0.0',           # RotY
            '0.0', '0.0', '0.0', '0.0',           # RotZ
            '0', '0', '0', '0', '0', '0',         # orientation counts
            fmt('StepCount', 0),
            fmt('LUX'), fmt('UVindex'),
            fmt('HighVisRaw'), fmt('LowVisRaw'), fmt('IRRaw'), fmt('UVRaw'),
            fmt('PMMeasCnt', 30),
            fmt('PM1MC'), fmt('PM1MCVar'),
            fmt('PM2_5MC'), fmt('PM2_5MCVar'),
            fmt('PM4MC'), fmt('PM4MCVar'),
            fmt('PM10MC'), fmt('PM10MCVar'),
            fmt('PM0_5NC'), fmt('PM0_5NCVar'),
            fmt('PM1NC'), fmt('PM1NCVar'),
            fmt('PM2_5NC'), fmt('PM2_5NCVar'),
            fmt('PM4NC'), fmt('PM4NCVar'),
            fmt('PM10NC'), fmt('PM10NCVar'),
            fmt('PMtypicalParticleSize'), fmt('PMtypicalParticleSizeVar'),
            f"{sampled_vol[i] * 1000:.3f}",       # PM2_5SampledMass (ug)
            fmt('PCB1T'), fmt('PCB2T'), fmt('FdpT'), fmt('AccelT'),
            '0.0',                                 # PT100R
            fmt('PCB2T'),                          # PCB2P ~ PCB2T
            '50.0', '50.0',                        # PumpPow1/2
            '3.7',                                 # PumpV
            fmt('MassFlow'), '1.5', '0.0',         # MassFlow, MFSVout, BFGenergy
            fmt('BattVolt', 4.1),
            '3.3', '5.0',                          # v3_3, v5
            '1', '0',                              # PumpsON, Dead
            '1', '1', '0',                         # BCS1, BCS2, BC_NPG
            '1', '1',                              # FLOWCTL, GPSRT
            '1', '1',                              # SD_DATAW, SD_HEADW
            '0', str(i * 30),                      # TPumpsOFF, TPumpsON
            fmt('CO2'), fmt('SCDT'), fmt('SCDRH'),
            fmt('VOCRaw'), fmt('NOXRaw'),
        ]
        rows.append(','.join(values))

    out_path = Path(output_dir) / fname
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with open(out_path, 'w') as f:
        f.write(header)
        f.write('\n')
        f.write('\n'.join(rows))
    return out_path

# ── edge cases ────────────────────────────────────────────────────────────────
def inject_upas_edge_cases(df, rng):
    # PM negative
    neg_idx = rng.integers(10, len(df) - 10)
    df.loc[neg_idx, 'PM2_5MC'] = -0.1

    # low battery
    batt_start = rng.integers(100, len(df) - 50)
    df.loc[batt_start:batt_start+20, 'BatteryCharge'] = 15

    # sustained PM zeros (filter obstruction)
    zero_start = rng.integers(200, len(df) - 50)
    for col in ['PM1MC', 'PM2_5MC', 'PM4MC', 'PM10MC']:
        df.loc[zero_start:zero_start+25, col] = 0.0

    # CO2 warmup spike
    df.loc[0:5, 'CO2'] = 1500.0

    return df

# ── campaign ──────────────────────────────────────────────────────────────────
def generate_upas_campaign(real_data_dir, output_dir,
                            n_sensors=3, n_days=7,
                            start='2022-09-15 08:00:00',
                            gps_utc_offset=-7.0,
                            seed=42, force_refit=False):
    real_data_dir, output_dir = validate_dirs(real_data_dir, output_dir,
                                               ext='.txt')
    model_path  = MODELS_DIR / "upas_synthesizer.pkl"
    synthesizer = (fit_upas(real_data_dir)
                   if force_refit or not model_path.exists()
                   else load_upas())

    n_records = n_days * 24 * 120   # 30s resolution
    outputs   = []

    for i in range(n_sensors):
        serial       = f"{99000 + i + 1:05d}"   # 99001, 99002 ...
        sample_name  = f"SYNTH{i+1:03d}"
        cartridge_id = f"C{99000 + i + 1}"
        rng          = np.random.default_rng(seed + i)

        synthetic = synthesizer.sample(num_rows=n_records)

        # force numeric — SDV can return object dtype
        synthetic = synthetic.apply(pd.to_numeric, errors='coerce').fillna(0.0)

        # clip to plausible ranges
        synthetic['PM1MC']   = synthetic['PM1MC'].clip(0, 500)
        synthetic['PM2_5MC']  = synthetic['PM2_5MC'].clip(0, 500)
        synthetic['PM4MC']    = synthetic['PM4MC'].clip(0, 500)
        synthetic['PM10MC']   = synthetic['PM10MC'].clip(0, 500)
        synthetic['AtmoT']    = synthetic['AtmoT'].clip(-5, 50)
        synthetic['AtmoRH']   = synthetic['AtmoRH'].clip(0, 100)
        synthetic['AtmoP']    = synthetic['AtmoP'].clip(950, 1050)
        synthetic['CO2']      = synthetic['CO2'].clip(400, 5000)
        synthetic['BatteryCharge'] = synthetic['BatteryCharge'].clip(0, 100)

        synthetic = inject_upas_edge_cases(synthetic, rng)

        out = write_upas(synthetic, serial, start, n_days,
                          output_dir, sample_name, cartridge_id,
                          gps_utc_offset)
        outputs.append(out)
        print(f"✓ Sensor {i+1}/{n_sensors}: {out.name}")

    print(f"\n✓ Generated {n_sensors} sensors × {n_days} days "
          f"({n_records} records each)")
    return outputs

# ── usage ─────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    generate_upas_campaign(
        real_data_dir  = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/upas",
        output_dir     = "/Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/upas",
        n_sensors      = 3,
        n_days         = 7,
        start          = "2022-09-15 08:00:00",
        gps_utc_offset = -7.0,
        seed           = 42
    )

✓ Found 3 real files in /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_raw/upas
✓ Output directory ready: /Users/markcampmier/Library/Mobile Documents/com~apple~CloudDocs/aerlift/data/0_synthetic/upas
Loading cached UPAS model from synthetic/models/upas_synthesizer.pkl
✓ Sensor 1/3: PSP99001_LOG_2022-09-15T08_00_00ZUTC_SYNTH001    _C99001    _.txt
✓ Sensor 2/3: PSP99002_LOG_2022-09-15T08_00_00ZUTC_SYNTH002    _C99002    _.txt
✓ Sensor 3/3: PSP99003_LOG_2022-09-15T08_00_00ZUTC_SYNTH003    _C99003    _.txt

✓ Generated 3 sensors × 7 days (20160 records each)


In [77]:
# write a test file and check
synthesizer = load_synthesizer('/Users/markcampmier/synthetic-data/synthetic/models/upas_synthesizer.pkl')

test_out = write_upas(
    synthesizer.sample(num_rows=10),
    serial='99001',
    start='2022-09-15 08:00:00',
    n_days=7,
    output_dir='/tmp'
)
print(find_data_start(test_out))  # should return 101

ValueError: Unknown format code 'f' for object of type 'str'